# 01. SDOF Ground Truth Verification
**Closed-Form Analytical & Independent Numerical Benchmarking**

Validates the OpenSeesPy SDOF solver against:
1. Closed-form underdamped free vibration analytical solution
2. Closed-form harmonic base excitation solution
3. Independent hand-coded Newmark-$\beta$ solver in NumPy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.ground_truth.opensees_sdof_model import SDOFParams, simulate_sdof
from src.ground_truth.independent_solvers import (
    analytical_sdof_free_vibration,
    newmark_nonlinear_sdof,
)
from src.evaluation.metrics import compute_rel_l2_error

# 1. Closed-form free vibration check
time = np.linspace(0, 5.0, 1001)
u_ana, v_ana, env = analytical_sdof_free_vibration(u0=0.05, v0=0.0, omega_n=2*np.pi/0.5, zeta=0.05, time=time)

params = SDOFParams(T=0.5, zeta=0.05, mass=1.0)
res_ops = simulate_sdof(params, ag=np.zeros_like(time), dt=0.005, u0=0.05, v0=0.0)

err = compute_rel_l2_error(res_ops.u[:len(time)], u_ana)
print(f"Closed-form free vibration relative L2 error: {err:.4f}% (< 0.01% target)")


In [ ]:
# 2. Plot Bilinear Hysteresis Loop vs Independent Newmark Solver
time = np.linspace(0, 10.0, 1000)
ag = 0.4 * 9.80665 * np.sin(2 * np.pi * 2.0 * time) * np.exp(-0.3 * time)

res_nm = newmark_nonlinear_sdof(mass=1.0, k0=1.0*(4*np.pi**2), zeta=0.05, ag=ag, dt=0.01, material_type="bilinear", u_y=0.01, alpha=0.05)

params_bi = SDOFParams(T=1.0, zeta=0.05, material_type="bilinear", u_y=0.01, alpha=0.05)
res_ops_bi = simulate_sdof(params_bi, ag=ag, dt=0.01)

plt.figure(figsize=(6, 5))
plt.plot(res_ops_bi.u * 1000, res_ops_bi.f_r, "r--", label="OpenSeesPy C++ NLTHA", lw=1.5)
plt.plot(res_nm["u"] * 1000, res_nm["f_r"], "b-", label="Independent NumPy Newmark-$\beta$", alpha=0.7, lw=1.2)
plt.axvline(10, color="k", linestyle=":", label="$\pm u_y$ (10 mm)")
plt.axvline(-10, color="k", linestyle=":")
plt.xlabel("Displacement $u(t)$ (mm)")
plt.ylabel("Restoring Force $F_R(t)$ (N)")
plt.title("Bilinear Kinematic Hardening Cross-Check")
plt.legend()
plt.grid(True)
plt.show()
